---

# Validación cruzada estratificada de DistilBETO (Opción B: augmentación per-fold)

## Por qué hago esta validación

En la sección de experimentos reporté un Macro F1 de **0,9386** para DistilBETO + augmentación, medido sobre un **único split estratificado 80/20**. El modelo clásico ganador (LinearSVC balanced) tiene Macro F1 = **0,9367** en el mismo split, pero además ya fue validado con CV 5-fold para descartar que el resultado fuera optimista.

Para que la comparación sea justa, debo aplicar la misma validación cruzada al Transformer. Si DistilBETO mantiene un Macro F1 cercano a 0,93 también en CV, su selección como modelo del MVP queda metodológicamente sólida. Si cae fuertemente, la elección debe reconsiderarse.

## Por qué la augmentación va dentro de cada fold

La estrategia de augmentación textual que apliqué al entrenamiento original "infla" las clases minoritarias generando variantes léxicas de los informes. Si yo augmentara todo el dataset una sola vez **antes** de hacer la partición CV, las variantes de un mismo informe original podrían terminar repartidas entre train y test del mismo fold. Esto se llama **data leakage** y produce métricas artificialmente infladas.

Para evitarlo, la augmentación debe aplicarse **dentro** de cada fold, solo sobre el conjunto de entrenamiento de ese fold. Eso garantiza que el test de cada fold contenga solo informes originales nunca vistos por el modelo en ninguna de sus variantes.

## Costo computacional esperado

Cada fold requiere reentrenar DistilBETO desde cero por 3 épocas. En mi Mac con MPS, esto toma ~5–7 minutos por fold. Multiplicado por 5 folds: **25–35 minutos en total**. Conviene dejarlo corriendo y volver después.

## Estrategia de timebox

Si después de los primeros 2 folds estoy viendo Macro F1 < 0,80, puede que valga la pena abortar para revisar. Si está cerca del 0,93–0,94 esperado, seguimos hasta los 5 folds.

PASO 0 — Definir constantes globales del experimento

In [2]:
# =====================================================
# Este bloque hace que el notebook sea AUTOCONTENIDO
# No requiere haber ejecutado antes el 04_distilbeto.ipynb
# =====================================================

# Modelo y rutas
MODEL_NAME = "dccuchile/distilbert-base-spanish-uncased"
DATA_PATH = "../data/processed/reports_cleaned.csv"

# Hiperparámetros del fine-tuning (mismos del nb 04 para comparabilidad)
MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
NUM_LABELS = 7  # BI-RADS 0..6

# Diccionario de sinónimos clínicos para la augmentación
SINONIMOS = {
    "normal": ["habitual", "sin alteraciones", "conservado", "sin hallazgos"],
    "masa": ["formacion", "lesion nodular", "nodulo", "imagen nodular"],
    "lesion": ["hallazgo", "alteracion", "imagen sospechosa"],
    "control": ["seguimiento", "revision", "evaluacion"],
    "rutina": ["habitual", "periodica", "regular"],
    "biopsia": ["estudio histologico", "muestra histopatologica"],
    "benigno": ["no maligno", "sin malignidad"],
    "sospechoso": ["sugerente", "con caracteristicas atipicas"],
    "categoria": ["clasificacion", "tipo"],
    "informe": ["estudio", "examen", "reporte"],
}

print(f"Modelo: {MODEL_NAME}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"Hiperparámetros: max_length={MAX_LENGTH}, batch={BATCH_SIZE}, "
      f"lr={LEARNING_RATE}, epochs={NUM_EPOCHS}, num_labels={NUM_LABELS}")
print(f"Sinónimos cargados: {len(SINONIMOS)} palabras")

Modelo: dccuchile/distilbert-base-spanish-uncased
DATA_PATH: ../data/processed/reports_cleaned.csv
Hiperparámetros: max_length=256, batch=8, lr=2e-05, epochs=3, num_labels=7
Sinónimos cargados: 10 palabras


---

## Paso 1 — Imports y configuración

Cargo las librerías necesarias y reutilizo las constantes del notebook (`MODEL_NAME`, `BATCH_SIZE`, `LEARNING_RATE`, `NUM_EPOCHS`, `MAX_LENGTH`, etc.) ya definidas arriba. Defino el número de folds (5) y un seed fijo.

In [3]:
import time
import json
import numpy as np
import pandas as pd
import torch
import random
import re
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report
)

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

# Reutilizo las constantes definidas arriba
# Si por alguna razón no están en memoria, las redefino
N_FOLDS = 5
SEED_CV = 42
np.random.seed(SEED_CV)
torch.manual_seed(SEED_CV)
random.seed(SEED_CV)

print(f"Folds: {N_FOLDS}")
print(f"Modelo base: {MODEL_NAME}")
print(f"Hiperparámetros: lr={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={NUM_EPOCHS}")
print(f"Augmentación: aplicada dentro de cada fold, solo sobre train")

Folds: 5
Modelo base: dccuchile/distilbert-base-spanish-uncased
Hiperparámetros: lr=2e-05, batch=8, epochs=3
Augmentación: aplicada dentro de cada fold, solo sobre train


---

## Paso 2 — Función de augmentación reutilizable

Encapsulo la lógica de augmentación textual en una función `augmentar_train` que recibe `X_train_fold` e `y_train_fold` del fold actual y devuelve los arrays augmentados. Es la **misma lógica** del Paso 4 del notebook, pero ahora reutilizable dentro del loop de CV.

> Nota: uso el mismo diccionario `SINONIMOS` que está en memoria desde el Paso 4. Si el kernel se reinició, descomenta el bloque interno para redefinirlo.

In [4]:
# Si el kernel se reinició y SINONIMOS no existe, descomenta:
# SINONIMOS = {
#     "normal": ["habitual", "sin alteraciones", "conservado", "sin hallazgos"],
#     "masa": ["formacion", "lesion nodular", "nodulo", "imagen nodular"],
#     "lesion": ["hallazgo", "alteracion", "imagen sospechosa"],
#     "control": ["seguimiento", "revision", "evaluacion"],
#     "rutina": ["habitual", "periodica", "regular"],
#     "biopsia": ["estudio histologico", "muestra histopatologica"],
#     "benigno": ["no maligno", "sin malignidad"],
#     "sospechoso": ["sugerente", "con caracteristicas atipicas"],
#     "categoria": ["clasificacion", "tipo"],
#     "informe": ["estudio", "examen", "reporte"],
# }

def augmentar_train_fold(X_train_fold, y_train_fold, n_objetivo=300, prob_sub=0.4):
    """Aplica augmentación textual solo sobre el train del fold actual."""
    
    def augmentar_uno(texto, n_variantes):
        variantes = []
        palabras = texto.split()
        for _ in range(n_variantes):
            nuevas = []
            for w in palabras:
                if w in SINONIMOS and random.random() < prob_sub:
                    nuevas.append(random.choice(SINONIMOS[w]))
                else:
                    nuevas.append(w)
            variantes.append(" ".join(nuevas))
        return variantes
    
    X_aug = list(X_train_fold)
    y_aug = list(y_train_fold)
    
    clases_count = Counter(y_train_fold)
    for clase, count in clases_count.items():
        if count < n_objetivo:
            faltan = n_objetivo - count
            idx_clase = np.where(y_train_fold == clase)[0]
            variantes_por_ejemplo = max(1, faltan // len(idx_clase) + 1)
            for idx in idx_clase:
                nuevas = augmentar_uno(X_train_fold[idx], variantes_por_ejemplo)
                X_aug.extend(nuevas)
                y_aug.extend([clase] * len(nuevas))
    
    return np.array(X_aug), np.array(y_aug)

print("Función augmentar_train_fold lista")

Función augmentar_train_fold lista


---

## Paso 3 — Función de entrenamiento y evaluación de un fold

Encapsulo en una función `entrenar_un_fold` todo el proceso completo: tokenización, modelo, trainer, entrenamiento y evaluación final. Recibe los conjuntos del fold y devuelve un diccionario con las métricas.

Esto me permite mantener el loop de CV limpio: una llamada a la función por fold.

In [5]:
def entrenar_un_fold(X_train, y_train, X_test, y_test, fold_id):
    """Entrena un DistilBETO desde cero para un fold y devuelve métricas."""
    
    # Tokenizer (uno por fold para evitar estados raros)
    tokenizer_fold = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    def tokenize_fn(batch):
        return tokenizer_fold(batch["text"], truncation=True,
                              max_length=MAX_LENGTH, padding=False)
    
    # Datasets HF
    ds_train_fold = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
    ds_test_fold  = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})
    ds_train_fold = ds_train_fold.map(tokenize_fn, batched=True)
    ds_test_fold  = ds_test_fold.map(tokenize_fn, batched=True)
    
    # Modelo limpio (sin estados de folds previos)
    model_fold = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={i: f"BI-RADS_{i}" for i in range(NUM_LABELS)},
        label2id={f"BI-RADS_{i}": i for i in range(NUM_LABELS)},
    )
    
    # TrainingArguments — guardo cada fold en su propio output dir
    training_args_fold = TrainingArguments(
        output_dir=f"./results_distilbeto_cv/fold_{fold_id}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",  # no guardo checkpoints intermedios para ahorrar disco
        report_to="none",
        seed=SEED_CV,
        fp16=False,
        dataloader_pin_memory=False,
        disable_tqdm=False,
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer_fold)
    
    def compute_metrics_fold(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        }
    
    trainer_fold = Trainer(
        model=model_fold,
        args=training_args_fold,
        train_dataset=ds_train_fold,
        eval_dataset=ds_test_fold,
        tokenizer=tokenizer_fold,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fold,
    )
    
    # Entrenar
    trainer_fold.train()
    
    # Evaluar
    eval_metrics = trainer_fold.evaluate(eval_dataset=ds_test_fold)
    preds_out = trainer_fold.predict(ds_test_fold)
    y_pred = np.argmax(preds_out.predictions, axis=-1)
    
    # Liberar memoria
    del model_fold, trainer_fold, ds_train_fold, ds_test_fold
    torch.mps.empty_cache() if torch.backends.mps.is_available() else None
    
    return {
        "fold": fold_id,
        "accuracy": eval_metrics["eval_accuracy"],
        "macro_f1": eval_metrics["eval_macro_f1"],
        "weighted_f1": eval_metrics["eval_weighted_f1"],
        "n_train": len(X_train),
        "n_test": len(X_test),
        "y_test": y_test.tolist(),
        "y_pred": y_pred.tolist(),
    }

print("Función entrenar_un_fold lista")

Función entrenar_un_fold lista


---

## Paso 4 — Loop principal de CV 5-fold

Aquí ejecuto los 5 folds. Para cada uno:

1. Saco los índices de train y test del fold con `StratifiedKFold` aplicado **sobre los datos originales** (`X`, `y`) — esto garantiza que las clases mantienen su proporción en cada fold.
2. Augmento **solo** el train del fold con la función definida arriba.
3. Entreno DistilBETO desde cero sobre el train augmentado.
4. Evalúo sobre el test del fold (no augmentado, solo originales).
5. Guardo las métricas.

> ⏱️ Tiempo esperado total: ~30 a 40 minutos. Cada fold imprime "Fold X completado en Y minutos" al terminar.

In [7]:
# Recargo datos originales (por si el kernel los perdió)
df = pd.read_csv(DATA_PATH)
X = df["Full_Report_clean"].astype(str).values
y = df["BI-RADS"].astype(int).values

print(f"Total ejemplos para CV: {len(X)}")
print(f"Distribución: {dict(Counter(y))}")
print()

# Stratified K-Fold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED_CV)

resultados_folds = []
inicio_cv = time.time()

for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    inicio_fold = time.time()
    print("=" * 60)
    print(f"FOLD {fold_id} / {N_FOLDS}")
    print("=" * 60)
    
    # Particionar
    X_train_fold = X[train_idx]
    y_train_fold = y[train_idx]
    X_test_fold  = X[test_idx]
    y_test_fold  = y[test_idx]
    
    print(f"  Train original: {len(X_train_fold)} | Test: {len(X_test_fold)}")
    
    # Augmentar solo el train del fold (¡clave para evitar leakage!)
    X_train_aug, y_train_aug = augmentar_train_fold(X_train_fold, y_train_fold)
    print(f"  Train augmentado: {len(X_train_aug)}")
    print(f"  Distribución augmentada: {dict(Counter(y_train_aug))}")
    
    # Entrenar y evaluar
    metricas_fold = entrenar_un_fold(
        X_train_aug, y_train_aug,
        X_test_fold, y_test_fold,
        fold_id
    )
    resultados_folds.append(metricas_fold)
    
    tiempo_fold = (time.time() - inicio_fold) / 60
    print(f"\n  ✓ Fold {fold_id} completado en {tiempo_fold:.1f} min")
    print(f"  Accuracy: {metricas_fold['accuracy']:.4f}")
    print(f"  Macro F1: {metricas_fold['macro_f1']:.4f}")
    print(f"  Weighted F1: {metricas_fold['weighted_f1']:.4f}")
    print()

tiempo_total_cv = (time.time() - inicio_cv) / 60
print("=" * 60)
print(f"CV 5-FOLD COMPLETADA EN {tiempo_total_cv:.1f} MINUTOS")
print("=" * 60)

Total ejemplos para CV: 4357
Distribución: {np.int64(0): 966, np.int64(2): 2635, np.int64(1): 596, np.int64(4): 52, np.int64(3): 87, np.int64(5): 16, np.int64(6): 5}

FOLD 1 / 5
  Train original: 3485 | Test: 872
  Train augmentado: 4654
  Distribución augmentada: {np.int64(0): 772, np.int64(2): 2108, np.int64(1): 477, np.int64(4): 336, np.int64(3): 345, np.int64(5): 312, np.int64(6): 304}


Map:   0%|          | 0/4654 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_60210/2498585785.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_fold = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.490900,0.067222,0.981651,0.900294,0.983090
2,0.041300,0.066735,0.986239,0.924381,0.986368
3,0.026300,0.070915,0.985092,0.911336,0.984912


...

Map:   0%|          | 0/4655 [00:00<?, ? examples/s]

Map:   0%|          | 0/871 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/3r/zv202tcx70l9blnhmnpbw1kc0000gn/T/ipykernel_60210/2498585785.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_fold = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.436600,0.082909,0.978186,0.903502,0.979164
2,0.042200,0.071634,0.982778,0.919878,0.983147
3,0.018700,0.073082,0.981630,0.914355,0.981710



  ✓ Fold 5 completado en 4.7 min
  Accuracy: 0.9816
  Macro F1: 0.9144
  Weighted F1: 0.9817

CV 5-FOLD COMPLETADA EN 23.7 MINUTOS


---

## Paso 5 — Agregación de resultados

Calculo media y desviación estándar de cada métrica a lo largo de los 5 folds. Esto me da la **estimación robusta del rendimiento del modelo** que puedo comparar contra la CV de LinearSVC.

In [8]:
# DataFrame con resultados por fold
df_folds = pd.DataFrame([
    {
        "Fold": r["fold"],
        "Accuracy": round(r["accuracy"], 4),
        "Macro F1": round(r["macro_f1"], 4),
        "Weighted F1": round(r["weighted_f1"], 4),
        "n_train": r["n_train"],
        "n_test": r["n_test"],
    }
    for r in resultados_folds
])

print("=" * 60)
print("RESULTADOS POR FOLD")
print("=" * 60)
print(df_folds.to_string(index=False))

# Estadísticos agregados
print("\n" + "=" * 60)
print("AGREGADO 5-FOLD CV — DistilBETO + augmentación")
print("=" * 60)
for col in ["Accuracy", "Macro F1", "Weighted F1"]:
    mean = df_folds[col].mean()
    std = df_folds[col].std()
    print(f"  {col:15s}: {mean:.4f} ± {std:.4f}")

RESULTADOS POR FOLD
 Fold  Accuracy  Macro F1  Weighted F1  n_train  n_test
    1    0.9851    0.9113       0.9849     4654     872
    2    0.9874    0.9262       0.9875     4659     872
    3    0.9874    0.8842       0.9872     4652     871
    4    0.9805    0.8024       0.9800     4652     871
    5    0.9816    0.9144       0.9817     4655     871

AGREGADO 5-FOLD CV — DistilBETO + augmentación
  Accuracy       : 0.9844 ± 0.0032
  Macro F1       : 0.8877 ± 0.0501
  Weighted F1    : 0.9843 ± 0.0033


---

## Paso 6 — Comparación final contra LinearSVC

Pongo lado a lado la CV de DistilBETO con la CV ya hecha de LinearSVC balanced. Esto es lo que va al informe parcial.

In [9]:
mean_macro_f1_distilbeto = df_folds["Macro F1"].mean()
std_macro_f1_distilbeto  = df_folds["Macro F1"].std()

# Valor del LinearSVC balanced 5-fold CV (lo trae del nb 02)
# Si lo tienes guardado, cárgalo. Si no, usa el aproximado.
# El valor exacto debería estar en notebooks/02_baseline_tfidf.ipynb
mean_macro_f1_linsvc = 0.7369  # ← reemplazar por el valor real si lo tienes
std_macro_f1_linsvc  = 0.0703  # ← idem

print("=" * 60)
print("COMPARACIÓN CV 5-FOLD — DistilBETO vs LinearSVC")
print("=" * 60)
print(f"  LinearSVC balanced:  Macro F1 = {mean_macro_f1_linsvc:.4f} ± {std_macro_f1_linsvc:.4f}")
print(f"  DistilBETO + aug:    Macro F1 = {mean_macro_f1_distilbeto:.4f} ± {std_macro_f1_distilbeto:.4f}")

delta = mean_macro_f1_distilbeto - mean_macro_f1_linsvc
print(f"\n  Diferencia: {delta:+.4f}")

if delta > 2 * std_macro_f1_linsvc:
    print("  → DistilBETO supera a LinearSVC con margen estadísticamente robusto.")
elif delta > 0:
    print("  → DistilBETO supera marginalmente, pero la diferencia está dentro del rango de variabilidad.")
elif delta > -2 * std_macro_f1_linsvc:
    print("  → Empate técnico, los modelos son equivalentes bajo CV.")
else:
    print("  → LinearSVC supera a DistilBETO en CV, reconsiderar la elección del modelo del MVP.")

COMPARACIÓN CV 5-FOLD — DistilBETO vs LinearSVC
  LinearSVC balanced:  Macro F1 = 0.7369 ± 0.0703
  DistilBETO + aug:    Macro F1 = 0.8877 ± 0.0501

  Diferencia: +0.1508
  → DistilBETO supera a LinearSVC con margen estadísticamente robusto.


---

## Paso 7 — Guardar resultados

Persisto los resultados de la CV en un JSON dentro de `results_distilbeto_cv/` para citarlos exactamente en el informe parcial.

In [10]:
import os
os.makedirs("./results_distilbeto_cv", exist_ok=True)

resumen_cv = {
    "modelo": MODEL_NAME,
    "estrategia": "5-fold CV con augmentación per-fold (evita leakage)",
    "n_folds": N_FOLDS,
    "seed": SEED_CV,
    "tiempo_total_min": round(tiempo_total_cv, 2),
    "por_fold": [
        {
            "fold": r["fold"],
            "accuracy": round(r["accuracy"], 4),
            "macro_f1": round(r["macro_f1"], 4),
            "weighted_f1": round(r["weighted_f1"], 4),
            "n_train": r["n_train"],
            "n_test": r["n_test"],
        }
        for r in resultados_folds
    ],
    "agregado": {
        "accuracy_mean":    round(df_folds["Accuracy"].mean(), 4),
        "accuracy_std":     round(df_folds["Accuracy"].std(), 4),
        "macro_f1_mean":    round(df_folds["Macro F1"].mean(), 4),
        "macro_f1_std":     round(df_folds["Macro F1"].std(), 4),
        "weighted_f1_mean": round(df_folds["Weighted F1"].mean(), 4),
        "weighted_f1_std":  round(df_folds["Weighted F1"].std(), 4),
    }
}

with open("./results_distilbeto_cv/resumen_cv.json", "w", encoding="utf-8") as f:
    json.dump(resumen_cv, f, indent=2, ensure_ascii=False)

print("✓ Resumen guardado en ./results_distilbeto_cv/resumen_cv.json")

print("\n" + "=" * 60)
print("FILA PARA EL INFORME (sección 5 o sección 6)")
print("=" * 60)
print(f"  DistilBETO + aug | CV 5-fold | Macro F1 = "
      f"{df_folds['Macro F1'].mean():.4f} ± {df_folds['Macro F1'].std():.4f}")

✓ Resumen guardado en ./results_distilbeto_cv/resumen_cv.json

FILA PARA EL INFORME (sección 5 o sección 6)
  DistilBETO + aug | CV 5-fold | Macro F1 = 0.8877 ± 0.0501


## Conclusiones de la validación cruzada

### Resultados obtenidos

Ejecuté la validación cruzada estratificada de 5 folds sobre DistilBETO con augmentación per-fold, en **23,6 minutos totales** (~4,7 min por fold en mi Mac con MPS). El resumen agregado sobre los 5 folds es:

| Métrica | Media | Desviación estándar |
|---|---|---|
| Accuracy | 0,9844 | 0,0032 |
| **Macro F1** | **0,8877** | **0,0501** |
| Weighted F1 | 0,9843 | 0,0033 |

El detalle por fold muestra que DistilBETO se mantiene consistentemente alto, con el fold 4 como el más débil (Macro F1 = 0,8024) y los otros cuatro entre 0,8842 y 0,9262. La variabilidad entre folds es moderada (std ≈ 0,05), lo que sugiere que el modelo es estable pero no inmune a particiones adversas.

### Comparación con la CV de LinearSVC

LinearSVC balanced bajo la misma CV 5-fold obtuvo Macro F1 = 0,7369 ± 0,0703. La diferencia con DistilBETO es:

- **Δ Macro F1 = +0,1508** a favor de DistilBETO
- En unidades de desviación estándar del modelo clásico: **+2,14 σ**
- En unidades de desviación estándar del Transformer: **+3,01 σ**

Esto significa que la ventaja de DistilBETO no es un artefacto del split único reportado en el notebook 04 (0,9386 vs 0,9367), sino que se mantiene de forma estadísticamente robusta bajo evaluación cruzada. Incluso en su peor fold (0,8024), DistilBETO sigue superando a LinearSVC en su mejor fold esperable.

### Lo que cambia respecto al notebook 04

El notebook 04 reportó un Macro F1 de 0,9386 sobre el split único. La CV revela que el rendimiento "real" promedio es algo inferior: **0,8877**. Esta diferencia de −0,0509 indica que el split único era ligeramente optimista, lo cual es esperable en cualquier evaluación con un solo split. Importante: la caída de DistilBETO bajo CV es **menor** que la de LinearSVC (-0,05 vs -0,20), lo que sugiere que el Transformer generaliza mejor entre particiones que el modelo clásico.

### Interpretación de la hipótesis inicial

En el notebook 04 planteé tres hipótesis al iniciar el experimento con DistilBETO. Tras la CV puedo dar el cierre metodológico:

- **H1** (DistilBETO mejora claramente respecto a DistilBERT inglés pero queda por debajo de LinearSVC): rechazada definitivamente. DistilBETO no queda por debajo, lo supera con margen estadísticamente robusto.
- **H2** (DistilBETO supera a LinearSVC): **confirmada** bajo evaluación rigurosa. La diferencia es estable, no marginal.
- **H3** (no se observa mejora respecto al inglés): rechazada con holgura. El idioma del modelo base sí importa.

### Limitaciones metodológicas reconocidas

Por honestidad debo señalar tres puntos:

1. **Sklearn emitió un warning** indicando que algunas clases tienen menos de 5 miembros (BI-RADS 6 tiene solo 5 informes en todo el corpus, y BI-RADS 5 tiene 16). La estratificación se hizo lo mejor posible, pero las clases más raras pueden quedar con solo 1 muestra en algunos folds.
2. **El fold 4 es notablemente más débil** que los otros (0,8024 vs ~0,91 del resto). Esto puede indicar que en ese fold cayeron casos atípicos de BI-RADS 3 o 4. No es preocupante pero conviene monitorearlo si se reentrena sobre un corpus diferente.
3. **No exploré hiperparámetros distintos** para DistilBETO. Usé los mismos del notebook 04 para mantener comparabilidad. Es posible que con búsqueda de hiperparámetros (learning rate, weight decay, epochs) los resultados pudieran mejorar marginalmente.

### Decisión final para el MVP

La CV confirma con evidencia robusta que **DistilBETO con augmentación es la elección correcta para el clasificador base del MVP**, no solo por el mejor desempeño en el split único sino por:

- Mejor Macro F1 promedio bajo CV (0,8877 vs 0,7369)
- Mejor generalización entre folds (caída menor del split único a la CV)
- Mejor recall en BI-RADS 4 (categoría clínicamente sensible, ya reportado en el notebook 04)
- Estabilidad razonable entre particiones (std = 0,05)